In [305]:
from IPython.display import Audio
import numpy as np
import matplotlib.pyplot as plt
SR = 48000

In [306]:
def play(wave, sample_rate=SR):
    return Audio(wave, rate=sample_rate, normalize=False)

def envelope_exp(env_duration=5, tau=0.35, sample_rate=SR):
    if tau <= 0:
        raise ValueError("tau must be positive.")
    n_env = round(env_duration * sample_rate)
    env = [np.exp(-(i / sample_rate) / tau) for i in range(n_env)]

    return env

env = envelope_exp()



In [307]:
Amplitude = 1
Duration = 2
N = int(Duration * SR)
Time = np.linspace(0, Duration,N, endpoint=False)

Frequencies = [262, 524, 786, 1048, 1310, 1572]

Amplitudes = [1.0, 0.15, 0.02, 0.005, 0.0025, 0.00125]

taus = [0.9, 0.055, 0.045, 0.035, 0.025, 0.02]

In [308]:
Freq_Do_triangle = 262

Init_duration = 5
N_global = int(SR * Init_duration)
T_global = np.linspace(0, Init_duration, N_global, endpoint=False) 
Amplitude_global = 0.5
Do_Triwave = np.zeros(N_global)
# the basic quantities I need to make a sign wave

depth = 5
period = 0.25
rate = 1 / period
# the Vibrato parameters
# depth = maximum frequency deviation in Hz
# the period of vibrato, in seconds
# rate how msny cycles every second in Hz


# for putting the periodic quantity in the sine, it must be:
# np.sin(2 * np.pi * freq * T_global - (depth / rate) * np.cos(2 * np.pi * rate * T_global)) 

# so: 

for freq, amp in zip(Frequencies, Amplitudes):
    Do_triangle = amp * (2/np.pi) * np.arcsin(
        np.sin(
            2 * np.pi * freq * T_global
            - (depth / rate) * np.cos(2 * np.pi * rate * T_global)
        )
    )

# originan phase = 2 * np.pi * freq * T_global
# Vibrato Modulation = - (depth / rate) * np.cos(2 * np.pi * rate * T_global)
# where:
# np.cos(...) creates the periodic modulation
# 2 * np.pi * rate * T_global : determines the rate of modulation
# depth / rate : determines the amplitude of the modulation
# I divide the depth with the rate cause I want it to express the 
# change of frequency in Hz.

    Do_Triwave += Do_triangle

Do_Triwave *= Amplitude_global
# *= means x = x * amplitude

Do = Audio(Do_Triwave, rate=SR, autoplay=False)
display(Do)

In [309]:
Do_tri_env = Do_Triwave * env
play(Do_tri_env)

In [310]:
#### Ι WANT TO MAKE A CHORUS ####

# it will pan around creating the sound of the moving snake


In [311]:
# Chorus parameters
depth = 0.003  # modulation depth in seconds
rate = 0.8  # modulation rate in Hz

# small depth value and small rate value: distinct chorus
# large depth value and small rate value : a lot of pitch movement
# large depth, large rate : weird shit 
# small depth, large rate : vibrato

t = np.arange(N_global) / SR

# Modulated delay
delay = depth * SR * np.sin(2 * np.pi * rate * t)

# Delayed signal using interpolation
indices = np.arange(len(Do_tri_env)) - delay

# Chorus Signal
Chorus_signal = np.interp(
    indices,
    np.arange(len(Do_tri_env)),
    Do_tri_env
)

In [312]:
play(Chorus_signal)

In [313]:
# 0.004 depth and 4 rate : DEATH SOUND
#          

In [314]:
###### NOW DO THIS FOR ALL NOTES #####

In [315]:
# First, our glossary

Notes = {
    "Do4": 262.00,
    "Do#4": 277.58,
    "Re4": 294.33,
    "Re#4": 311.82,
    "Mi4": 330.22,
    "Fa4": 349.89,
    "Fa#4": 370.69,
    "Sol4": 392.99,
    "Sol#4": 416.36,
    "La4": 441.00,
    "La#4": 467.21,
    "Si4": 494.99,
    "Do5": 524.00,
    "Do#5": 555.16,
    "Re5": 588.66,
    "Re#5": 623.64,
    "Mi5": 660.44,
    "Fa5": 699.78,
    "Fa#5": 741.38,
    "Sol5": 785.98,
    "Sol#5": 832.72,
    "La5": 882.00,
    "La#5": 934.42,
    "Si5": 989.98,
    "Do6": 1048.00
}

In [316]:
class Tri_note:

    def __init__(self, note_name):

        # Get the fundamental frequency
        self.frequency = Notes[note_name]

        # Define harmonic frequencies
        self.Frequencies = [
            self.frequency * 1,
            self.frequency * 2,
            self.frequency * 3,
            self.frequency * 4,
            self.frequency * 5,
            self.frequency * 6
        ]

        # Define harmonic amplitudes
        self.Amplitudes = [1.0, 0.15, 0.02, 0.005, 0.0025, 0.00125]

        # Define vibrato parameters
        self.depth = 5
        self.period = 0.25
        self.rate = 1 / self.period

        # Define the global time vector
        self.N_global = int(SR * Init_duration)
        self.T_global = np.linspace(
            0, Init_duration, self.N_global, endpoint=False
        )

        # Initialize the waveform
        self.Do_Triwave = np.zeros(self.N_global)

        # Generate and sum the harmonic components
        for freq, amp in zip(self.Frequencies, self.Amplitudes):

            Do_triangle = amp * (2/np.pi) * np.arcsin(
                np.sin(
                    2 * np.pi * freq * self.T_global
                    - (self.depth / self.rate)
                    * np.cos(2 * np.pi * self.rate * self.T_global)
                )
            )

            self.Do_Triwave += Do_triangle

        # Apply global amplitude
        self.Do_Triwave *= Amplitude_global

        # Apply amplitude envelope
        self.Do_tri_env = self.Do_Triwave * env


                # Define chorus parameters
        self.chorus_depth = 0.003
        self.chorus_rate = 0.8

        # Generate time vector
        self.t = np.arange(len(self.Do_tri_env)) / SR

        # Calculate delay modulation
        self.delay = self.chorus_depth * SR * np.sin(
            2 * np.pi * self.chorus_rate * self.t
        )

        # Calculate interpolation indices
        self.indices = np.arange(len(self.Do_tri_env)) - self.delay

         # Apply chorus using global modulation
        self.Chorus_signal = np.interp(
            indices,
            np.arange(len(self.Do_tri_env)),
            self.Do_tri_env
        )

In [317]:
Do4 = Tri_note("Do4")
play(Do4.Chorus_signal)

In [318]:
# Create all note objects
Tri_notes = {
    note_name: Tri_note(note_name)
    for note_name in Notes
}

In [324]:
play(Tri_notes["Do4"].Chorus_signal)
play(Tri_notes["Do6"].Chorus_signal)

In [320]:
# Stereo channels
panL = -0.7
panR = 0.7

Left = Do_tri_env * (1 - panL) / 2
Right = Chorus_signal * (1 + panR) / 2

Stereo = np.vstack((Left, Right)).astype(np.float32)

# vertical stack: takes the two arrays and puts them on top of each other
# astype(np.float32): converts the data into float32 numbers,
# appropriate for our signal

Do = Audio(Stereo, rate=SR, autoplay=False)
display(Do)